# 地方競馬 〜 ◎（1着予想）に絞った単勝の買い方（3年分・保存はしない）

◎は的中率が高く、締切直前のオッズの下がり方も小さい（本番の記録: 3倍未満の勝ち馬は確定までに約4%、5〜20倍は約2割下がる）。
◎・○・▲ごとに、期待値×1着率の細かい表を2024〜2026年の各年後半で出し、**本番の目減りを差し引いた回収率**も並べる。

- モデルは本番と同じ（F4込み、乱数3回の平均）。判断も払戻も確定オッズ（＋目減りを差し引いた参考値）
- 目減りの目安（本番の記録の勝ち馬の中央値）: 3倍未満 0.96 / 3〜5倍 0.88 / 5〜10倍 0.81 / 10〜20倍 0.83 / 20倍以上 0.80

**Input**: 学習用と同じ3つ。Internet は不要。**実行**: Save & Run All (Commit)。1時間前後。

## ⓪ 設定

In [ ]:
import os, glob, warnings
import numpy as np, pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)

DATA_ROOT   = "/kaggle/input"
TEST_YEARS  = list(range(2016, 2026))  # 1年ずつ評価する年（各年とも「前年まで」のデータだけで学習）
BASE_END    = "2015-01-01"   # スピード指数の基準タイムはこの日より前のデータだけで作る（全ての評価年より過去）
USE_ODDS    = True          # True: オッズも特徴量に入れ、「市場の評価のどこがズレているか」を直接学習させる

## ① 読込（オッズ "2,012.0" のカンマで割れた行を直して読む）

In [ ]:
DIR = os.path.dirname(glob.glob(f"{DATA_ROOT}/**/keiba_results.csv", recursive=True)[0])

def fix_row(row):
    if len(row) == 19:
        return row[:15] + [row[15] + row[16]] + row[17:]
    return None

res = pd.read_csv(f"{DIR}/keiba_results.csv", engine="python", on_bad_lines=fix_row,
                  dtype={"race_id": str, "horse_id": str, "jockey_id": str})
races = pd.read_csv(f"{DIR}/keiba_races.csv", dtype={"race_id": str})
pays  = pd.read_csv(f"{DIR}/keiba_payouts.csv", dtype={"race_id": str, "horse_num": str})
# 追加データ（2026年収集分・過去の穴埋め分）。収集ノートブックの出力を Private データセットにして Add Input しておくと自動で結合
for tag in ["2026", "fill"]:
    add = {n: sorted(glob.glob(f"{DATA_ROOT}/**/keiba_{n}_{tag}.csv", recursive=True)) for n in ["results", "races", "payouts"]}
    if not all(add.values()):
        continue
    res = pd.concat([res] + [pd.read_csv(p, dtype={"race_id": str, "horse_id": str, "jockey_id": str}) for p in add["results"]],
                    ignore_index=True).drop_duplicates(["race_id", "number"], keep="last")
    races = pd.concat([races] + [pd.read_csv(p, dtype={"race_id": str}) for p in add["races"]],
                      ignore_index=True).drop_duplicates("race_id", keep="last")
    pays = pd.concat([pays] + [pd.read_csv(p, dtype={"race_id": str, "horse_num": str}) for p in add["payouts"]],
                     ignore_index=True).drop_duplicates(keep="last")
    print(f"追加データ（{tag}）を結合しました:", add["races"])
print("results:", res.shape, "/ races:", races.shape, "/ payouts:", pays.shape)

## ② 地方に絞り、race_id から日付を作る

地方の race_id は「年(4) + 場(2) + 月(2) + 日(2) + R(2)」。帯広（ばんえい, 場コード65）は別競技なので除外。
下の確認で「月」が 1〜12 だけ、「日」が 1〜31 だけになっていれば解釈は正しい。

In [ ]:
EXCLUDE_VENUES = {"65"}

def is_nar(rid):
    code = rid.str[4:6]
    return ~code.astype(int).between(1, 10) & ~code.isin(EXCLUDE_VENUES)

res   = res[is_nar(res["race_id"])].copy()
races = races[is_nar(races["race_id"])].copy()

mm = races["race_id"].str[6:8].astype(int)
dd = races["race_id"].str[8:10].astype(int)
print("月の範囲:", mm.min(), "〜", mm.max(), "/ 日の範囲:", dd.min(), "〜", dd.max())
print("月ごとのレース数:", mm.value_counts().sort_index().to_dict())
assert mm.between(1, 12).all() and dd.between(1, 31).all(), "月日として解釈できない値があります"

races["date"] = pd.to_datetime(races["race_id"].str[:4] + races["race_id"].str[6:10],
                               format="%Y%m%d", errors="coerce")
print("日付が作れなかったレース:", races["date"].isna().sum())
races = races.dropna(subset=["date"])
pays  = pays[pays["race_id"].isin(set(races["race_id"]))].copy()
print("地方レース数:", len(races), "/ 期間:", races["date"].min().date(), "〜", races["date"].max().date())

## ③ 整形

In [ ]:
race_cols = ["race_id", "date", "venue", "race_number", "course_type", "distance",
             "turn", "weather", "track_condition"]
df = res.merge(races[race_cols], on="race_id", how="inner")

# 着順: 取消・除外は出走していないので除く / 中止・失格は最下位扱い / "2(降)" は数字部分
df = df[~df["rank"].astype(str).isin(["取", "除"])].copy()
df["n_runners"] = df.groupby("race_id")["race_id"].transform("size")
df["finish"] = pd.to_numeric(df["rank"].astype(str).str.extract(r"^(\d+)")[0], errors="coerce")
df["finish"] = df["finish"].fillna(df["n_runners"]).astype(int)
df["win"]  = (df["finish"] == 1).astype(int)
df["top3"] = (df["finish"] <= 3).astype(int)
df["finish_ratio"] = (df["finish"] - 1) / (df["n_runners"] - 1).clip(lower=1)

# 数値化
for c in ["frame", "number", "weight", "distance", "race_number", "odds", "popularity", "last_3f"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df["sex"] = df["sex_age"].astype(str).str[0]
df["age"] = pd.to_numeric(df["sex_age"].astype(str).str[1:], errors="coerce")
bw = df["horse_weight"].astype(str).str.extract(r"(\d+)\(([-+]?\d+)\)")
df["body_weight"] = pd.to_numeric(bw[0], errors="coerce")
df["bw_diff"]     = pd.to_numeric(bw[1], errors="coerce")
t = df["time"].astype(str).str.extract(r"(?:(\d+):)?(\d+\.\d)")
df["time_sec"] = pd.to_numeric(t[0], errors="coerce").fillna(0) * 60 + pd.to_numeric(t[1], errors="coerce")
pos = df["passing"].astype(str).str.split("-")
df["first_corner"] = pd.to_numeric(pos.str[0], errors="coerce")
df["horse_key"]  = df["horse_id"].fillna(df["horse_name"])
df["jockey_key"] = df["jockey_id"].fillna(df["jockey_name"])
df["month"] = df["date"].dt.month

# レース後にしか分からない「そのレースでの内容」（前走特徴量の材料。そのレースの特徴量には使わない）
df["time_behind"]  = df["time_sec"] - df.groupby("race_id")["time_sec"].transform("min")
df["last3f_rank"]  = df.groupby("race_id")["last_3f"].rank(pct=True)
df["corner1_ratio"] = df["first_corner"] / df["n_runners"]
print(df.shape)

## ④ 過去走の特徴量（必ず shift(1) で当該レースを除外）

スピード指数は「直近3年の基準タイム」と「その日の馬場差」で計算（2026/10 改良）。

F4（メンバーの中での位置: 5走平均・最高指数のメンバー内の偏差、最上位との差、メンバー上位3頭の平均との差）も追加（2026/11）。

In [ ]:
# --- スピード指数: 競馬場・コース・距離・馬場ごとの基準タイムとの差（50が平均、高いほど速い）---
# 基準タイムは BASE_END より前の5着以内の馬だけで作る（どの評価年から見ても過去のデータ）
BASE_KEYS = ["venue", "course_type", "distance", "track_condition"]
base = (df[(df["date"] < BASE_END) & (df["finish"] <= 5)]
        .groupby(BASE_KEYS)["time_sec"].agg(base_mean="mean", base_std="std", base_n="size").reset_index())
base = base[base["base_n"] >= 30]
df = df.drop(columns=["base_mean", "base_std", "base_n", "speed"], errors="ignore").merge(base, on=BASE_KEYS, how="left")
df["speed"] = (df["base_mean"] - df["time_sec"]) / df["base_std"] * 10 + 50
print("スピード指数が計算できた割合:", round(df["speed"].notna().mean(), 3))

# --- スピード指数の作り直し（2026/10 改良：比較実験で5回全勝）---
# 1) 基準タイムを「各年について、その前の3年」で作り直す（馬場の改修や新しい距離設定に追随。未来の年は使わない）
# 2) その日・その競馬場・コースの時計の速さ（馬場差）で補正する（レース後に分かる値で、次走以降の特徴量にだけ使う）
df["year"] = df["date"].dt.year
df["_bk"] = (df["venue"].astype(str) + "|" + df["course_type"].astype(str) + "|" + df["distance"].astype(str)
             + "|" + df["track_condition"].astype(str))
parts = []
for Y in sorted(df["year"].unique()):
    s_ = df[(df["year"] >= Y - 3) & (df["year"] <= Y - 1) & (df["finish"] <= 5)]
    st = s_.groupby("_bk")["time_sec"].agg(b2_mean="mean", b2_std="std", b2_n="size").reset_index()
    st = st[st["b2_n"] >= 30]
    st["year"] = Y
    parts.append(st)
df = df.drop(columns=["b2_mean", "b2_std", "b2_n"], errors="ignore").merge(pd.concat(parts, ignore_index=True), on=["_bk", "year"], how="left")
raw2 = ((df["b2_mean"] - df["time_sec"]) / df["b2_std"] * 10 + 50).fillna(df["speed"])
race_dev = raw2.where(df["finish"] <= 5).groupby(df["race_id"]).transform("mean") - 50
day = df["date"].astype(str) + "_" + df["venue"].astype(str) + "_" + df["course_type"].astype(str)
first = ~df["race_id"].duplicated()
day_dev = race_dev[first].groupby(day[first]).mean()
n_day = first.groupby(day).sum()
variant = day.map(day_dev).where(day.map(n_day) >= 3)
df["speed_old"] = df["speed"]
df["speed"] = raw2 - variant.fillna(0)
print("新しいスピード指数: 直近3年の基準が使えた割合", round(df["b2_mean"].notna().mean(), 3),
      "/ 馬場差で補正できた割合", round(variant.notna().mean(), 3), "/ 計算できた割合", round(df["speed"].notna().mean(), 3))

# 予想用ノートブックで結果を手入力したときに使う基準タイム（直近3年。馬場差の補正は無し）
LY = int(df["year"].max())
base = (df[(df["year"] >= LY - 2) & (df["finish"] <= 5)]
        .groupby(BASE_KEYS, observed=True)["time_sec"].agg(base_mean="mean", base_std="std", base_n="size").reset_index())
base = base[base["base_n"] >= 30]

df = df.sort_values(["date", "race_id", "number"]).reset_index(drop=True)
g = df.groupby("horse_key", sort=False)

def past_mean(col, n):
    s = g[col].shift(1)
    return s.groupby(df["horse_key"]).rolling(n, min_periods=1).mean().reset_index(level=0, drop=True)

for k in (1, 2, 3):
    df[f"prev_finish_{k}"] = g["finish"].shift(k)
df["prev_finish_ratio"]   = g["finish_ratio"].shift(1)
df["prev_speed"]          = g["speed"].shift(1)
df["avg_speed_5"]         = past_mean("speed", 5)
df["max_speed_5"]         = (g["speed"].shift(1).groupby(df["horse_key"]).rolling(5, min_periods=1).max()
                             .reset_index(level=0, drop=True))
df["avg_finish_ratio_5"]  = past_mean("finish_ratio", 5)
df["top3_rate_5"]         = past_mean("top3", 5)
df["win_rate_10"]         = past_mean("win", 10)
df["prev_time_behind"]    = g["time_behind"].shift(1)
df["avg_time_behind_5"]   = past_mean("time_behind", 5)
df["prev_last3f_rank"]    = g["last3f_rank"].shift(1)
df["avg_last3f_rank_5"]   = past_mean("last3f_rank", 5)
df["prev_corner1_ratio"]  = g["corner1_ratio"].shift(1)
df["avg_corner1_ratio_5"] = past_mean("corner1_ratio", 5)
df["prev_popularity"]     = g["popularity"].shift(1)
df["prev_log_odds"]       = np.log1p(g["odds"].shift(1))
df["n_races"]             = g.cumcount()
df["days_since_last"]     = (df["date"] - g["date"].shift(1)).dt.days
df["dist_diff"]           = df["distance"] - g["distance"].shift(1)
df["weight_diff"]         = df["weight"] - g["weight"].shift(1)
df["same_venue_as_prev"]  = (df["venue"] == g["venue"].shift(1)).astype(float)

# 騎手・調教師の通算成績（前日までの成績のみ。平滑化つき）
K = 30
def past_rate(key, target):
    daily = (df.groupby([key, "date"])[target].agg(["sum", "count"])
               .reset_index().sort_values([key, "date"]))
    gg = daily.groupby(key)
    past_sum = gg["sum"].cumsum() - daily["sum"]
    past_cnt = gg["count"].cumsum() - daily["count"]
    p = df[target].mean()
    daily[f"{key}_{target}_rate"] = (past_sum + p * K) / (past_cnt + K)
    return daily[[key, "date", f"{key}_{target}_rate"]]

for key, target in [("jockey_key", "top3"), ("jockey_key", "win"), ("trainer", "top3")]:
    df = df.merge(past_rate(key, target), on=[key, "date"], how="left")

# レース内での相対位置（メンバーの中で強いか）
for c in ["avg_finish_ratio_5", "top3_rate_5", "avg_time_behind_5", "jockey_key_top3_rate",
          "avg_speed_5", "max_speed_5"]:
    df[f"{c}_inrace"] = df.groupby("race_id")[c].rank(pct=True)

# F4 メンバーの中での位置（2026/11 改良：3年×乱数10回の検証で26勝4敗）
for c in ["avg_speed_5", "max_speed_5"]:
    gc = df.groupby("race_id")[c]
    df[f"{c}_z"] = (df[c] - gc.transform("mean")) / gc.transform("std").replace(0, np.nan)
df["max_speed_5_gap"] = df["max_speed_5"] - df.groupby("race_id")["max_speed_5"].transform("max")
rk = df.groupby("race_id")["max_speed_5"].rank(ascending=False, method="first")
df["field_level"] = df["max_speed_5"].where(rk <= 3).groupby(df["race_id"]).transform("mean")
df["max_speed_5_vs_field"] = df["max_speed_5"] - df["field_level"]
F4_COLS = ["avg_speed_5_z", "max_speed_5_z", "max_speed_5_gap", "field_level", "max_speed_5_vs_field"]

cat_cols = ["venue", "course_type", "turn", "weather", "track_condition", "sex"]
for c in cat_cols:
    df[c] = df[c].astype("category")

features = (
    ["frame", "number", "weight", "age", "body_weight", "bw_diff", "distance", "race_number",
     "n_runners", "month"] + cat_cols +
    ["prev_finish_1", "prev_finish_2", "prev_finish_3", "prev_finish_ratio", "avg_finish_ratio_5",
     "prev_speed", "avg_speed_5", "max_speed_5",
     "top3_rate_5", "win_rate_10", "prev_time_behind", "avg_time_behind_5", "prev_last3f_rank",
     "avg_last3f_rank_5", "prev_corner1_ratio", "avg_corner1_ratio_5", "prev_popularity",
     "prev_log_odds", "n_races", "days_since_last", "dist_diff", "weight_diff", "same_venue_as_prev",
     "jockey_key_top3_rate", "jockey_key_win_rate", "trainer_top3_rate"] +
    [c for c in df.columns if c.endswith("_inrace")] + F4_COLS
)
if USE_ODDS:
    features.append("odds")
print(len(features), "features")

## ⑤ 各年のモデルを作り、後半の予想を集める

In [ ]:
from scipy.optimize import minimize
import time

SEEDS = [42, 43, 44]
df["year"] = df["date"].dt.year
inv = 1 / df["odds"].where(df["odds"] > 0)
df["p_mkt"] = inv / inv.groupby(df["race_id"]).transform("sum")
df["race_ok"] = (df.groupby("race_id")["p_mkt"].transform(lambda s: s.notna().all())
                 & (df.groupby("race_id")["win"].transform("sum") == 1))
LAST = int(df["year"].max())
YEARS = [LAST, LAST - 1, LAST - 2]
order = ["date", "race_id", "number"]
params = dict(n_estimators=3000, learning_rate=0.05, num_leaves=63, min_child_samples=100,
              subsample=0.8, subsample_freq=1, colsample_bytree=0.8, verbose=-1)

def comb(d, w):
    s = w[0] * np.log(d["pm"].clip(1e-6)) + w[1] * np.log(d["p_mkt"].clip(1e-6))
    e = np.exp(s - s.groupby(d["race_id"]).transform("max"))
    return e / e.groupby(d["race_id"]).transform("sum")

def ll(w, d):
    return -np.log(comb(d, w)[d["win"] == 1].clip(1e-9)).mean()

def race_norm(x, rid):
    x = pd.Series(np.asarray(x, dtype=float), index=rid.index)
    return (x / x.groupby(rid).transform("sum")).values

COLS = ["race_id", "date", "number", "win", "odds", "p_mkt"]
PARTS = []
for Y in YEARS:
    t0 = time.time()
    tr = df[df["year"] <= Y - 1].sort_values(order)
    v_ = df[(df["year"] == Y) & df["race_ok"]].sort_values(order)
    mid = v_["date"].drop_duplicates().sort_values().iloc[v_["date"].nunique() // 2]
    va_fit, va_ev = v_[v_["date"] < mid], v_[v_["date"] >= mid]
    pf, pe = [], []
    for sd in SEEDS:
        m = lgb.LGBMClassifier(**{**params, "random_state": sd})
        m.fit(tr[features], tr["win"], eval_set=[(va_fit[features], va_fit["win"])], eval_metric="binary_logloss",
              callbacks=[lgb.early_stopping(100, verbose=False)])
        pf.append(race_norm(m.predict_proba(va_fit[features])[:, 1], va_fit["race_id"]))
        pe.append(race_norm(m.predict_proba(va_ev[features])[:, 1], va_ev["race_id"]))
    F = va_fit[COLS].assign(pm=np.mean(pf, axis=0))
    E = va_ev[COLS].assign(pm=np.mean(pe, axis=0))
    w = minimize(ll, x0=[0.5, 0.5], args=(F,), method="Nelder-Mead").x
    E["p"] = comb(E, w).values
    E["ev"] = E["p"] * E["odds"]
    E["年"] = Y
    PARTS.append(E)
    print(f"{Y}年: 学習 〜{Y - 1}年 / 後半（採点）{E['race_id'].nunique():,}レース（{mid.date()}〜、{E['date'].nunique()}日）"
          f" / 合成の重み モデル {w[0]:.3f}・市場 {w[1]:.3f} / 市場との差 {ll([0, 1], E) - ll(w, E):.5f} / {time.time() - t0:.0f}秒")
A = pd.concat(PARTS, ignore_index=True)

## ⑥ ◎・○・▲ ごとの成績

In [ ]:

A = A.sort_values(["race_id", "p"], ascending=[True, False])
A["印順"] = A.groupby("race_id").cumcount() + 1
A["印"] = A["印順"].map({1: "◎", 2: "○", 3: "▲"}).fillna("他")
A["ret"] = A["odds"] * A["win"]
DRIFT = [(0, 3, 0.96), (3, 5, 0.88), (5, 10, 0.81), (10, 20, 0.83), (20, 1e9, 0.80)]
A["drift"] = 1.0
for lo, hi, f in DRIFT:
    A.loc[(A["odds"] >= lo) & (A["odds"] < hi), "drift"] = f
A["ret_live"] = A["ret"] * A["drift"]
def fmt(s):
    return f"{s['ret'].mean():.0%}" if len(s) else "-"
print("【印ごとの全体（期待値の条件なし）】")
for m in ["◎", "○", "▲"]:
    s = A[A["印"] == m]
    print(f"  {m}: {len(s):,}頭 的中率 {s['win'].mean():.1%} / 回収率 {s['ret'].mean():.0%}（目減り後 {s['ret_live'].mean():.0%}）/ 平均オッズ {s['odds'].mean():.1f}")


In [ ]:

EV_E = [0, 0.9, 0.95, 1.0, 1.05, 1.1, 1.2, 1.3, 99]
EV_L = ["〜0.9", "0.9-0.95", "0.95-1.0", "1.0-1.05", "1.05-1.1", "1.1-1.2", "1.2-1.3", "1.3〜"]
P_E = [0, .2, .3, .4, .5, 1.01]
P_L = ["〜20%", "20-30%", "30-40%", "40-50%", "50%〜"]
A["期待値帯"] = pd.cut(A["ev"], EV_E, labels=EV_L, right=False)
A["1着率帯"] = pd.cut(A["p"], P_E, labels=P_L, right=False)
for m in ["◎", "○"]:
    s = A[A["印"] == m]
    g = s.groupby(["1着率帯", "期待値帯"], observed=False)
    n = g.size().unstack()
    r = (g["ret"].mean().unstack() * 100).round(0)
    rl = (g["ret_live"].mean().unstack() * 100).round(0)
    hit = (g["win"].mean().unstack() * 100).round(0)
    print(f"\n==================== {m} ====================")
    print("▼ 頭数"); print(n.to_string())
    print("▼ 的中率 %"); print(hit.to_string(na_rep="-"))
    print("▼ 回収率 %（確定オッズ）"); print(r.where(n >= 30).to_string(na_rep="-"))
    print("▼ 回収率 %（本番の目減りを差し引いた目安）"); print(rl.where(n >= 30).to_string(na_rep="-"))

print("\n【◎ で「期待値≧X」の買い方（1着率の条件ごと）】年ごとは " + " / ".join(map(str, YEARS)))
rows = []
fav = A[A["印"] == "◎"]
days = A["date"].nunique()
for pmin in [0.0, 0.2, 0.3, 0.4]:
    for evmin in [0.9, 0.95, 1.0, 1.05, 1.1, 1.2]:
        s = fav[(fav["ev"] >= evmin) & (fav["p"] >= pmin)]
        if len(s) < 50:
            continue
        yr = [s.loc[s["年"] == Y, "ret"].mean() for Y in YEARS]
        rows.append({"1着率≧": f"{pmin:.0%}", "期待値≧": evmin, "点数": len(s), "1日あたり": round(len(s) / days, 1),
                     "的中率": f"{s['win'].mean():.1%}", "回収率": f"{s['ret'].mean():.0%}", "目減り後": f"{s['ret_live'].mean():.0%}",
                     "年ごと": " / ".join(f"{y:.0%}" for y in yr), "3年とも100%超": "○" if all(y > 1 for y in yr) else ""})
print(pd.DataFrame(rows).to_string(index=False))

print("\n【比較: 今の案F（期待値1.2以上・1着率10%以上・1レース1頭、印を問わない）】")
s = A[(A["ev"] >= 1.2) & (A["p"] >= 0.10)]
s = s.loc[s.groupby("race_id")["ev"].idxmax()]
yr = [s.loc[s["年"] == Y, "ret"].mean() for Y in YEARS]
print(f"  {len(s)}点 的中率 {s['win'].mean():.1%} / 回収率 {s['ret'].mean():.0%}（目減り後 {s['ret_live'].mean():.0%}）/ 年ごと " + " / ".join(f"{y:.0%}" for y in yr))
print("  印の内訳: " + " / ".join(f"{k} {v}" for k, v in s["印"].value_counts().items()))
print("\n※ 目減り後は、本番の記録（数週間分）の勝ち馬のオッズの下がり方を当てはめた目安。記録が増えれば値も更新する")
